In [5]:
import os
import random
import shutil
import cv2
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

cv2.setNumThreads(0)

# ==========================================
# 1. CONFIGURARE CAI 
# ==========================================
# Sursa: datasetul pe care tocmai l-am curatat si impartit
INPUT_BASE_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_cleaned")

# Destinatia: un dataset complet nou, gata de antrenare
OUTPUT_BASE_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_augmented")

CLASSES = ["0", "1", "2", "3", "4"]
WORKERS = max((os.cpu_count() or 2) - 1, 1)

In [6]:
# ==========================================
# 2. FUNCTII DE AUGMENTARE FINA
# ==========================================
def apply_flip(image, rng):
    if rng.random() < 0.5: image = cv2.flip(image, 1)
    if rng.random() < 0.5: image = cv2.flip(image, 0)
    return image

def apply_affine(image, rng):
    h, w = image.shape[:2]
    angle = rng.uniform(-45, 45)
    scale = rng.uniform(0.85, 1.15) 
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))

def apply_blur(image, rng):
    return cv2.GaussianBlur(image, rng.choice([(3,3), (5,5)]), rng.uniform(0.3, 1.2))

def apply_color_jitter(image, rng):
    return cv2.convertScaleAbs(image, alpha=rng.uniform(0.85, 1.15), beta=rng.uniform(-10, 10))

def apply_noise(image, rng):
    row, col, ch = image.shape
    gauss = np.random.normal(0, rng.uniform(1, 5) ** 0.5, (row, col, ch)).astype('float32')
    noisy_image = cv2.add(image.astype('float32'), gauss)
    return np.clip(noisy_image, 0, 255).astype('uint8')

def dynamic_augment(image, seed):
    rng = random.Random(seed)
    ops_pool = [(apply_flip, "flip"), (apply_affine, "affine"), (apply_color_jitter, "color")]
    
    conflict = rng.choice(["blur", "noise", "none"])
    if conflict == "blur": ops_pool.append((apply_blur, "blur"))
    elif conflict == "noise": ops_pool.append((apply_noise, "noise"))
        
    chosen_ops = rng.sample(ops_pool, rng.randint(2, len(ops_pool)))
    
    applied_names = []
    for op_func, op_name in chosen_ops:
        image = op_func(image, rng)
        applied_names.append(f"aug_{op_name}")
        
    return image, "_".join(applied_names)

def process_task(task):
    task_type = task[0]
    
    if task_type == 'copy':
        _, src, dst = task
        shutil.copy2(src, dst)
        return True, ""
        
    elif task_type == 'augment':
        _, src_path, output_dir_cls, idx, seed = task
        img = cv2.imread(str(src_path))
        if img is None: return False, f"Eroare citire: {src_path.name}"
            
        augmented_image, aug_suffix = dynamic_augment(img, seed)
        noul_nume = f"{src_path.stem}_{idx:03d}_{aug_suffix}{src_path.suffix}"
        
        cv2.imwrite(str(output_dir_cls / noul_nume), augmented_image)
        return True, ""

print("Functiile de augmentare au fost definite cu succes.")

Functiile de augmentare au fost definite cu succes.


In [7]:
# ==========================================
# 3. EXECUTIA PRINCIPALA
# ==========================================
def main():
    if not INPUT_BASE_DIR.exists():
        print(f"Eroare: Folderul sursa {INPUT_BASE_DIR} nu exista!")
        return
        
    if OUTPUT_BASE_DIR.exists():
        print("Curatam folderul de output vechi...")
        shutil.rmtree(OUTPUT_BASE_DIR)

    rng = random.Random(42)
    tasks = []

    print("\n--- 1. Analiza setului curatat si pregatirea task-urilor ---")
    
    # Aflam targetul maxim din setul de Train
    train_input = INPUT_BASE_DIR / "train"
    if not train_input.exists():
        print("Folderul train lipseste din setul curatat!")
        return
        
    class_counts = {cls: len(list((train_input / cls).glob("*.*"))) for cls in CLASSES if (train_input / cls).exists()}
    target_train_count = max(class_counts.values()) if class_counts else 0
    print(f"Target echilibrare pentru Train: {target_train_count} imagini/clasa.")

    for split in ["train", "val", "test"]:
        for cls in CLASSES:
            input_dir_cls = INPUT_BASE_DIR / split / cls
            output_dir_cls = OUTPUT_BASE_DIR / split / cls
            output_dir_cls.mkdir(parents=True, exist_ok=True)
            
            if not input_dir_cls.exists(): continue
                
            originals = sorted(list(input_dir_cls.glob("*.*")))
            current_count = len(originals)
            if current_count == 0: continue
            
            # 1. Toate imaginile originale se copiaza în noul dataset
            for src_path in originals:
                tasks.append(('copy', src_path, output_dir_cls / src_path.name))
                
            # 2. Daca suntem in Train, calculam diferentele si generam augmentari
            if split == "train":
                missing = target_train_count - current_count
                if missing > 0:
                    for idx in range(missing):
                        src_path = originals[idx % current_count]
                        seed = rng.randint(0, 9999999)
                        tasks.append(('augment', src_path, output_dir_cls, idx + 1, seed))

    print(f"\n--- 2. Construim Noul Dataset ({len(tasks)} operatiuni)... ---")
    processed, errors = 0, 0

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = [executor.submit(process_task, task) for task in tasks]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Creare Dataset"):
            ok, msg = future.result()
            if ok: processed += 1
            else: 
                errors += 1
                print(f"\n[Eroare] {msg}")

    print("\n" + "="*50)
    print("FINALIZAT CU SUCCES!")
    print("="*50)
    print(f"Noul tau dataset echilibrat se afla in:\n📂 {OUTPUT_BASE_DIR}")
    
    # Afisam structura finala Train pentru validare
    print("\nValidare set Train nou:")
    for cls in CLASSES:
        count = len(list((OUTPUT_BASE_DIR / "train" / cls).glob("*.*")))
        print(f"Clasa {cls}: {count} imagini")

In [8]:
if __name__ == '__main__':
    main()

Curatam folderul de output vechi...

--- 1. Analiza setului curatat si pregatirea task-urilor ---
Target echilibrare pentru Train: 12906 imagini/clasa.

--- 2. Construim Noul Dataset (72789 operatiuni)... ---


Creare Dataset:   0%|          | 0/72789 [00:00<?, ?it/s]


FINALIZAT CU SUCCES!
Noul tau dataset echilibrat se afla in:
📂 B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_augmented

Validare set Train nou:
Clasa 0: 12906 imagini
Clasa 1: 12906 imagini
Clasa 2: 12906 imagini
Clasa 3: 12906 imagini
Clasa 4: 12906 imagini
